# Semana 13: Interpretabilidad de Modelos
## Notebook Conceptual (NB1) – Explicaciones con Modelos Sencillos

**Propósito:** Aprender a explicar modelos complejos, tanto global como localmente, cumpliendo con necesidades de negocio y regulatorias.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Calcular importancia de características (permutation importance) y comparar con la importancia del árbol.
- Dibujar Partial Dependence Plots (PDP) para una o dos variables.
- Usar SHAP para explicaciones locales y globales.
- Usar LIME para explicar una instancia específica.
- Comparar las explicaciones de SHAP y LIME.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, r2_score

# SHAP
import shap

# LIME
from lime.lime_tabular import LimeTabularExplainer

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga del Dataset: California Housing

El dataset **California Housing** contiene información sobre viviendas en distritos de California. El objetivo es predecir el valor mediano de las casas.

In [ ]:
# Cargamos el dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Dimensiones del dataset: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()

In [ ]:
# Descripción de las características
print("Características:")
for i, name in enumerate(housing.feature_names):
    print(f"{name}: {housing.DESCR.split('------')[1].split('Attributes:')[1].split('Attributes:')[1].split('\n')[i+2].strip()}")

In [ ]:
# Variable objetivo
y = housing.target
X = housing.data

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
y.hist(bins=50, edgecolor='black')
plt.title('Distribución de MedHouseVal')
plt.xlabel('Valor medio de la vivienda')

plt.subplot(1, 2, 2)
plt.boxplot(y)
plt.title('Boxplot de MedHouseVal')
plt.ylabel('Valor')

plt.tight_layout()
plt.show()

---
## 2. Entrenamiento de un Random Forest

Dividimos en train/test y entrenamos un Random Forest para tener un modelo complejo que explicar.

In [ ]:
# División en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenamos Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluación básica
y_pred = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

---
## 3. Importancia de Características

### 3.1. Importancia por impureza (Gini importance) del Random Forest

In [ ]:
# Importancia por impureza
importances_rf = rf.feature_importances_
feature_names = X.columns

imp_df = pd.DataFrame({
    'Característica': feature_names,
    'Importancia': importances_rf
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df, x='Importancia', y='Característica')
plt.title('Importancia de Características (Gini - Random Forest)')
plt.tight_layout()
plt.show()

print("Importancia por impureza:")
print(imp_df)

### 3.2. Importancia por permutación (más fiable)

In [ ]:
# Importancia por permutación
perm_importance = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

perm_df = pd.DataFrame({
    'Característica': feature_names,
    'Importancia': perm_importance.importances_mean,
    'Desviación': perm_importance.importances_std
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=perm_df, x='Importancia', y='Característica', xerr=perm_df['Desviación'])
plt.title('Importancia por Permutación')
plt.tight_layout()
plt.show()

print("Importancia por permutación:")
print(perm_df)

### 3.3. Comparación de ambas métricas

In [ ]:
comparacion = pd.DataFrame({
    'Característica': feature_names,
    'Gini Importance': importances_rf,
    'Permutation Importance': perm_importance.importances_mean
})

print("Comparación de importancias:")
comparacion.round(4)

---
## 4. Partial Dependence Plots (PDP)

Visualizamos cómo cambia la predicción al variar una o dos características.

In [ ]:
# PDP para una variable (MedInc - ingreso medio)
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(rf, X_train, ['MedInc'], ax=ax, grid_resolution=50)
plt.title('PDP - Ingreso Medio (MedInc)')
plt.show()

In [ ]:
# PDP para otra variable (AveOccup - ocupación promedio)
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(rf, X_train, ['AveOccup'], ax=ax, grid_resolution=50)
plt.title('PDP - Ocupación Promedio (AveOccup)')
plt.show()

In [ ]:
# PDP para dos variables (interacción)
fig, ax = plt.subplots(figsize=(10, 8))
PartialDependenceDisplay.from_estimator(rf, X_train, [('MedInc', 'AveOccup')], ax=ax, grid_resolution=20)
plt.title('PDP Interacción - MedInc vs AveOccup')
plt.show()

### 4.1. Individual Conditional Expectation (ICE)

Muestra las curvas individuales para cada observación.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
PartialDependenceDisplay.from_estimator(rf, X_train, ['MedInc'], ax=ax, 
                                         kind='individual', grid_resolution=50)
plt.title('ICE - Ingreso Medio (MedInc)')
plt.show()

---
## 5. SHAP (SHapley Additive exPlanations)

Usamos TreeExplainer para calcular valores SHAP.

In [ ]:
# Creamos el explainer
explainer = shap.TreeExplainer(rf)

# Calculamos SHAP para un subconjunto de test (para no saturar la memoria)
X_test_sample = X_test.sample(100, random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print("Valores SHAP calculados.")

### 5.1. Summary Plot (importancia global con dirección)

In [ ]:
plt.figure(figsize=(12, 6))
shap.summary_plot(shap_values, X_test_sample, feature_names=feature_names)
plt.show()

### 5.2. Dependence Plot (similar a PDP pero con SHAP)

In [ ]:
# Dependence plot para MedInc
shap.dependence_plot('MedInc', shap_values, X_test_sample, feature_names=feature_names)
plt.show()

In [ ]:
# Dependence plot con color de interacción (ej. con AveOccup)
shap.dependence_plot('MedInc', shap_values, X_test_sample, 
                     feature_names=feature_names, interaction_index='AveOccup')
plt.show()

### 5.3. Waterfall Plot (explicación local para una instancia)

In [ ]:
# Elegimos una instancia de ejemplo (índice 0 del sample)
instance_idx = 0
shap.waterfall_plot(shap.Explanation(values=shap_values[instance_idx], 
                                      base_values=explainer.expected_value,
                                      data=X_test_sample.iloc[instance_idx].values,
                                      feature_names=feature_names))
plt.show()

### 5.4. Force Plot (otra visualización local)

In [ ]:
# Force plot para la misma instancia
shap.initjs()  # para visualización interactiva en notebook
shap.force_plot(explainer.expected_value, shap_values[instance_idx], 
                X_test_sample.iloc[instance_idx], feature_names=feature_names, matplotlib=True)
plt.show()

---
## 6. LIME (Local Interpretable Model-agnostic Explanations)

Usamos LIME para explicar la misma instancia que con SHAP.

In [ ]:
# Creamos el explainer de LIME
lime_explainer = LimeTabularExplainer(
    X_train.values,
    feature_names=feature_names,
    class_names=['MedHouseVal'],
    mode='regression',
    random_state=42
)

# Instancia a explicar (misma que en SHAP)
instance = X_test_sample.iloc[instance_idx].values

# Explicación
lime_exp = lime_explainer.explain_instance(instance, rf.predict, num_features=len(feature_names))

# Mostrar explicación
print("Explicación LIME:")
lime_exp.show_in_notebook(show_table=True)

# Extraer coeficientes como DataFrame
lime_map = dict(lime_exp.as_list())
lime_df = pd.DataFrame(list(lime_map.items()), columns=['Característica', 'Efecto'])
print("\nEfectos LIME:")
print(lime_df)

---
## 7. Comparación de SHAP y LIME

Comparamos las explicaciones de SHAP y LIME para la misma instancia.

In [ ]:
# Obtenemos valores SHAP para la instancia
shap_instance = pd.Series(shap_values[instance_idx], index=feature_names)

# Creamos un DataFrame comparativo
comparacion_explicaciones = pd.DataFrame({
    'Característica': feature_names,
    'SHAP': shap_instance.values,
    'LIME': [lime_map.get(feature, 0) for feature in feature_names]
})

print("Comparación de valores SHAP y efectos LIME:")
comparacion_explicaciones.round(4)

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparacion_explicaciones.sort_values('SHAP', ascending=True).plot(
    x='Característica', y='SHAP', kind='barh', ax=axes[0], legend=False, color='blue')
axes[0].set_title('SHAP Values')
axes[0].set_xlabel('Contribución')

comparacion_explicaciones.sort_values('LIME', ascending=True).plot(
    x='Característica', y='LIME', kind='barh', ax=axes[1], legend=False, color='green')
axes[1].set_title('LIME Effects')
axes[1].set_xlabel('Contribución')

plt.tight_layout()
plt.show()

### Análisis de la comparación:

- **SHAP** proporciona valores teóricamente consistentes (basados en Shapley) y suma a la predicción menos el valor base.
- **LIME** es más rápido y agnóstico al modelo, pero puede ser inestable (la explicación varía con el muestreo).
- En general, SHAP es preferido para modelos basados en árboles por su rapidez y consistencia.
- Ambas herramientas son complementarias: SHAP para análisis riguroso, LIME para exploración rápida.

---
## 8. Conclusiones

Hemos explorado las principales técnicas de interpretabilidad de modelos:

✔️ **Importancia de características**:
- Gini importance (intrínseca al modelo) vs permutation importance (más fiable).
- La permutación muestra que algunas variables tienen importancia similar.

✔️ **PDP e ICE**:
- Visualizamos cómo afecta una variable a la predicción promedio y a nivel individual.
- La relación entre ingreso y precio es positiva y aproximadamente lineal.

✔️ **SHAP**:
- Proporciona explicaciones locales consistentes (waterfall, force plots).
- Summary plot muestra importancia global con dirección del efecto.
- Dependence plot revela interacciones.

✔️ **LIME**:
- Explicación local rápida mediante un modelo lineal en la vecindad.
- Útil para exploración, aunque menos estable que SHAP.

**Lección clave**: La interpretabilidad es esencial para entender, depurar y generar confianza en los modelos. SHAP es actualmente el estándar de facto, especialmente para modelos basados en árboles.

---
**Fin del Notebook Conceptual - Semana 13**